# 01 · Data audit
What is wrong with the raw Quikr file, what the cleaning pipeline does about it, and what was *not* touched. Every number below is read from files produced by the pipeline (`reports/metrics/`), not typed in.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "carprice").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
from carprice.config import load_settings
S = load_settings()
P = S.paths
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.float_format", "{:,.3f}".format)
def show_fig(name):
    display(Image(filename=str(P.figures / name), width=900))
def md(text): display(Markdown(text))


## 1. The raw file
Everything is read as text first so that no parsing decision is hidden in a type conversion.

In [2]:
raw = pd.read_csv(P.data_raw / "quikr_car.csv", dtype=str)
print("shape:", raw.shape)
display(raw.head(8))
rep = pd.DataFrame({"missing": raw.isna().sum(), "pct": (raw.isna().mean()*100).round(1), "n_unique": raw.nunique()})
display(rep)
print("exact duplicate rows:", int(raw.duplicated().sum()), f"({raw.duplicated().mean():.1%})")

shape: (892, 6)


,name,company,year,Price,kms_driven,fuel_type
0,Hyundai Santro Xing XO eRLX Euro III,Hyundai,2007,"80,000","45,000 kms",Petrol
1,Mahindra Jeep CL550 MDI,Mahindra,2006,"4,25,000",40 kms,Diesel
2,Maruti Suzuki Alto 800 Vxi,Maruti,2018,Ask For Price,"22,000 kms",Petrol
3,Hyundai Grand i10 Magna 1.2 Kappa VTVT,Hyundai,2014,"3,25,000","28,000 kms",Petrol
4,Ford EcoSport Titanium 1.5L TDCi,Ford,2014,"5,75,000","36,000 kms",Diesel
5,Ford EcoSport Titanium 1.5L TDCi,Ford,2015,Ask For Price,"59,000 kms",Diesel
6,Ford Figo,Ford,2012,"1,75,000","41,000 kms",Diesel
7,Hyundai Eon,Hyundai,2013,"1,90,000","25,000 kms",Petrol


,missing,pct,n_unique
name,0,0.000,525
company,0,0.000,48
year,0,0.000,61
Price,0,0.000,274
kms_driven,52,5.800,258
fuel_type,55,6.200,3


exact duplicate rows: 94 (10.5%)


## 2. Junk the original notebook only partly handled

In [3]:
bad_year = raw[~raw.year.str.fullmatch(r"\d{4}", na=False)]
md(f"**{len(bad_year)} rows ({len(bad_year)/len(raw):.1%}) have a non-year in `year`** (e.g. {bad_year.year.head(6).tolist()}): mis-scraped ad text where several columns are shifted.")
md(f"**`Price`** has {int((raw.Price=='Ask For Price').sum())} 'Ask For Price' rows (no target) and uses Indian digit grouping (`4,25,000`).")
md(f"**`company`** has {raw.company.nunique()} distinct values for ~25 real brands: " + ", ".join(sorted(raw.company.unique())[:18]) + ", …")
display(raw[~raw.company.isin(raw.company.value_counts().head(25).index)].head(6))
kms = raw.kms_driven.dropna()
md(f"**`kms_driven`**: {int((~kms.str.contains('kms')).sum())} rows hold a fuel type instead of kilometres (field shift); {int(raw.kms_driven.isna().sum())} missing.")

**50 rows (5.6%) have a non-year in `year`** (e.g. ['...', '150k', 'TOUR', '...', 'r 15', 'Zest']): mis-scraped ad text where several columns are shifted.

**`Price`** has 35 'Ask For Price' rows (no target) and uses Indian digit grouping (`4,25,000`).

**`company`** has 48 distinct values for ~25 real brands: 2012, 7, 9, Any, Audi, BMW, Chevrolet, Commercial, Datsun, Fiat, Force, Ford, Hindustan, Honda, Hyundai, I, Jaguar, Jeep, …

,name,company,year,Price,kms_driven,fuel_type
232,Land Rover Freelander 2 SE,Land,2015,"21,00,000","30,000 kms",Diesel
294,Yama,Yamaha,r 15,"55,000",NaN,NaN
360,selling car Ta,selling,Zest,Ask For Price,NaN,NaN
401,URJE,URJENT,SELL,"1,80,000",NaN,NaN
414,Swift Dzire Tour 27 Dec 2016 Regis,Swift,tion,"3,70,000",NaN,NaN
441,Used Commercial Maruti Omn,Used,arry,"1,50,000",NaN,NaN


**`kms_driven`**: 2 rows hold a fuel type instead of kilometres (field shift); 52 missing.

## 3. Cleaning report (rows before / after every operation)

In [4]:
rep = pd.read_csv(P.metrics / "cleaning_report.csv")
display(rep)
meta = json.load(open(P.metrics / "data_meta.json"))
md(f"**{meta['raw_rows']} raw rows → {meta['n_rows']} clean rows.** Reference (snapshot) year = {meta['reference_year']}.")

,step,rows_before,rows_after,rows_removed,note
0,raw,892,892,0,as loaded (all columns read as text)
1,recover shifted fuel/kms fields,892,892,0,2 rows had fuel type in kms column: fuel recov...
2,drop non-numeric year (mis-scraped ad text),892,842,50,NaN
3,drop missing price ('Ask For Price'),842,819,23,no target label -> unusable
4,drop implausible model year,819,819,0,valid range 1990..2019
5,drop unknown/non-car brand,819,819,0,brand not in the supported canonical list (e.g...
6,drop price outside hard limits,819,819,0,"10,000..50,000,000 INR"
7,drop exact duplicate listings,819,727,92,92 exact repeats of an earlier row; kept first...
8,kms: unrealistic -> set missing (rows kept),727,727,0,"0 above 600,000 km, 6 cars >= 3y old with < 10..."
9,drop clear price typos (> 10x family median),727,726,1,"2 flagged for review (> 6.0x), 1 dropped; vali..."


**892 raw rows → 726 clean rows.** Reference (snapshot) year = 2019.

## 4. Outlier review: investigated, not blindly removed
A price is dropped only if it is both far above its own model family *and* a clear typo. Expensive-but-plausible cars stay.

In [5]:
display(pd.read_csv(P.metrics / "outlier_review.csv"))

,source_row,name,company,year,kms_driven,price_inr,model_family,family_median_price,price_to_family_median,decision
0,510,Jaguar XE XE Portfolio,Jaguar,2016,8500,2800000,XE,"2,800,000.000",1.000,kept: plausible high-priced / rare vehicle
1,562,Mahindra XUV500 W6,Mahindra,2014,45000,8500003,XUV500,"800,000.000",10.630,dropped: price typo
2,719,Audi A3 Cabriolet 40 TFSI,Audi,2015,12516,3100000,A3,"3,100,000.000",1.000,kept: plausible high-priced / rare vehicle
3,768,Hindustan Motors Ambassador Classic Mark 4 – Befo,Hindustan Motors,1995,37000,750000,Ambassador,"90,000.000",8.330,kept: plausible high-priced / rare vehicle
4,838,Ford Endeavor 4x4 Thunder Plus,Ford,2019,9000,2900000,Endeavor,"2,900,000.000",1.000,kept: plausible high-priced / rare vehicle


## 5. Missingness after cleaning and schema validation

In [6]:
clean = pd.read_csv(P.data_interim / "cleaned.csv")
display(pd.read_csv(P.metrics / "missingness_raw_vs_clean.csv"))
from carprice.data.schema import validate_clean
validate_clean(clean.assign(kms_driven=clean.kms_driven.astype(float)), meta["reference_year"])
md("✅ Pandera schema validation passed (types, ranges, allowed fuel types, positive prices).")
display(clean.describe(include="all").T)

,column,n_missing,pct_missing
0,name,0,0.000
1,company,0,0.000
2,year,0,0.000
3,kms_driven,8,1.100
4,fuel_type,0,0.000
5,price_inr,0,0.000
6,kms_suspect,0,0.000
7,source_row,0,0.000


✅ Pandera schema validation passed (types, ranges, allowed fuel types, positive prices).

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
name,726,465,Honda City,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN
company,726,25,Maruti,202,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year,726.000,NaN,NaN,NaN,"2,012.320",4.120,"1,995.000","2,010.000","2,013.000","2,015.000","2,019.000"
kms_driven,718.000,NaN,NaN,NaN,"47,155.045","34,462.835",0.000,"28,000.000","42,000.000","59,000.000","400,000.000"
fuel_type,726,4,Petrol,392,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price_inr,726.000,NaN,NaN,NaN,"391,273.310","373,699.026","30,000.000","165,000.000","289,999.500","489,499.250","3,100,000.000"
kms_suspect,726,2,False,720,NaN,NaN,NaN,NaN,NaN,NaN,NaN
source_row,726.000,NaN,NaN,NaN,452.961,256.111,0.000,222.250,467.500,673.750,891.000


## 6. Dataset provenance and licence

In [7]:
prov = json.load(open(P.reports / "dataset_sources.json"))
display(pd.DataFrame(prov).T[["acquired_via","filename","raw_rows","sha256","retrieved_at"]])
md("**Declared licence / terms:** " + prov["quikr"]["declared_license"])
md("**Features that do NOT exist in this dataset** (and are therefore not used or invented): transmission, owner count, engine capacity, horsepower, mileage (kmpl), condition, location, listing date. See `docs/DATASET_REPORT.md` for alternatives.")

,acquired_via,filename,raw_rows,sha256,retrieved_at
quikr,url,quikr_car.csv,892,a3f056c1f18b857ffec870e6855f70712753163fe5b23d...,2026-10-31T15:34:29+00:00


**Declared licence / terms:** Not stated. The CSV is a scrape of public Quikr classified listings redistributed through the author's own repository. Treat as educational/portfolio use only; do not redistribute commercially. Verify terms before any commercial use.

**Features that do NOT exist in this dataset** (and are therefore not used or invented): transmission, owner count, engine capacity, horsepower, mileage (kmpl), condition, location, listing date. See `docs/DATASET_REPORT.md` for alternatives.

## 7. Original project vs. this pipeline
| | original notebook | this pipeline |
|---|---|---|
| rows kept | 815 (incl. ~96 exact duplicates) | see cleaning report (duplicates removed first) |
| split | `random_state` scanned over 10 seeds, best R² (0.82) reported | grouped + stratified, repeated CV, untouched test split |
| encoder | `OneHotEncoder` fitted on **all** rows before the split | all learned preprocessing inside the CV pipeline |
| outliers | one hard cut at ₹60 lakh | reviewed individually, only typos dropped |
| odometer 40 km on a 2006 car | kept | flagged → missing → imputed |